# dataclasses-replace-args — worked example 1: Apply a dict of overrides to a base config in one replace call

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclasses-replace-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`dataclasses.replace(base, **overrides)` returns a *new* dataclass with the named fields swapped and every other field copied from `base`. Because the overrides come in as keyword arguments, a dict of `{field: value}` can be splatted straight in with `**override_dict`. The base instance is never mutated, and `__post_init__` re-runs on the clone.

## Worked solution

**Goal:** given one base `TrainingArgs` and a single dict like `{'lr': 5e-4, 'epochs': 3}`, produce one new variant with exactly those fields changed.

1. **Why `replace` and not manual copy.** Writing `TrainingArgs(lr=..., batch_size=base.batch_size, ...)` forces you to restate every field and breaks the moment a field is added. `replace` copies untouched fields for you.
2. **Splat the dict.** `replace(base, **overrides)` is identical to writing each key as a keyword argument. So `replace(base, **{'lr': 5e-4, 'epochs': 3})` becomes `replace(base, lr=5e-4, epochs=3)`.
3. **Immutability check.** `replace` builds a brand-new object; `base` keeps its original values. We assert `variant is not base` and that `base.lr` is unchanged.
4. **Validation still fires.** The clone runs through `__post_init__`, so an illegal override (e.g. negative lr) raises `ValueError` exactly as constructing from scratch would.

The printed output confirms the overridden fields changed while the untouched `batch_size` was copied from base.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def make_variant(base, overrides):
    return replace(base, **overrides)

base = TrainingArgs()
variant = make_variant(base, {'lr': 5e-4, 'epochs': 3})
print('variant.lr =', variant.lr, 'variant.epochs =', variant.epochs)
print('variant.batch_size copied from base =', variant.batch_size)
print('base unchanged, base.lr =', base.lr, '| different object =', variant is not base)